# Tunisian Bac Math RAG — Scientific Validation

**Purpose:** End-to-end validation of the RAG pipeline for thesis documentation.  
**What this tests:** environment integrity, DB contents, two-stage retrieval quality, LLM generation correctness, syllabus guard.  
**What this does NOT do:** re-index, rebuild, or modify any data.  

---

## 0. Environment & Sanity Checks

Verify that the runtime has everything needed before loading heavy models.

In [1]:
import sys
import os
import time
from pathlib import Path

print(f"Python  : {sys.version}")
print(f"Platform: {sys.platform}")
print(f"CWD     : {os.getcwd()}")
print()

Python  : 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:29:10) [GCC 14.3.0]
Platform: linux
CWD     : /home/jupyter/tunisian-bac-math-rag



In [2]:
# ── Package import checks ────────────────────────────────────
_ok = []
_fail = []

for pkg_name, import_name in [
    ("chromadb",       "chromadb"),
    ("FlagEmbedding",  "FlagEmbedding"),
    ("torch",          "torch"),
    ("vertexai",       "vertexai"),
    ("google.cloud.storage", "google.cloud.storage"),
    ("streamlit",      "streamlit"),
]:
    try:
        mod = __import__(import_name)
        ver = getattr(mod, "__version__", "(no __version__)")
        _ok.append((pkg_name, ver))
    except ImportError as e:
        _fail.append((pkg_name, str(e)))

for name, ver in _ok:
    print(f"  OK   {name:30s} {ver}")
for name, err in _fail:
    print(f"  FAIL {name:30s} {err}")

if _fail:
    print("\n!! Some packages are missing. Install them before continuing.")
else:
    print(f"\nAll {len(_ok)} packages imported successfully.")

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_supp

  OK   chromadb                       1.4.1
  OK   FlagEmbedding                  (no __version__)
  OK   torch                          2.10.0+cu128
  OK   vertexai                       1.134.0
  OK   google.cloud.storage           (no __version__)
  OK   streamlit                      1.53.1

All 6 packages imported successfully.


In [3]:
# ── Project files & DB path ──────────────────────────────────
PROJECT_ROOT = Path(os.getcwd())

required_files = ["config.py", "rag_engine.py"]
for f in required_files:
    p = PROJECT_ROOT / f
    status = "EXISTS" if p.exists() else "MISSING"
    print(f"  {status:7s} {f}")

db_path = PROJECT_ROOT / "tunisian_math_db"
print(f"\n  DB dir exists : {db_path.exists()}")
if db_path.exists():
    items = list(db_path.iterdir())
    print(f"  DB dir items  : {len(items)}")
    for item in sorted(items)[:8]:
        print(f"    {item.name}")

  EXISTS  config.py
  EXISTS  rag_engine.py

  DB dir exists : True
  DB dir items  : 3
    ab88ffbd-d994-4bb7-a530-2ee6d41f0ce5
    chroma.sqlite3
    index_manifest.json


In [4]:
# ── Confirm config loads and show key values ─────────────────
import config

print("Config loaded successfully.")
print(f"  PROJECT_ID                : {config.PROJECT_ID}")
print(f"  REGION                    : {config.REGION}")
print(f"  LOCAL_DB_PATH             : {config.LOCAL_DB_PATH}")
print(f"  COLLECTION_NAME           : {config.COLLECTION_NAME}")
print(f"  EMBEDDING_MODEL_NAME      : {config.EMBEDDING_MODEL_NAME}")
print(f"  USE_FP16                  : {config.USE_FP16}")
print(f"  CHAT_MODEL_ID             : {config.CHAT_MODEL_ID}")
print(f"  RETRIEVE_K_FIRST_PASS     : {config.RETRIEVE_K_FIRST_PASS}")
print(f"  RETRIEVE_K_SECOND_PASS    : {config.RETRIEVE_K_SECOND_PASS}")
print(f"  USE_TOP_N                 : {config.USE_TOP_N}")
print(f"  SIMILARITY_GOOD_THRESHOLD : {config.SIMILARITY_GOOD_THRESHOLD}")
print(f"  SIMILARITY_FALLBACK_THRESH: {config.SIMILARITY_FALLBACK_THRESHOLD}")
print(f"  CHUNK_CORRECTION          : {config.CHUNK_CORRECTION}")
print(f"  CHUNK_TEXTBOOK            : {config.CHUNK_TEXTBOOK}")

Config loaded successfully.
  PROJECT_ID                : project-e9871d91-8923-44ed-887
  REGION                    : us-central1
  LOCAL_DB_PATH             : ./tunisian_math_db
  COLLECTION_NAME           : bac_math_exercises
  EMBEDDING_MODEL_NAME      : BAAI/bge-m3
  USE_FP16                  : False
  CHAT_MODEL_ID             : gemini-2.0-flash-exp
  RETRIEVE_K_FIRST_PASS     : 10
  RETRIEVE_K_SECOND_PASS    : 8
  USE_TOP_N                 : 6
  SIMILARITY_GOOD_THRESHOLD : 1.2
  SIMILARITY_FALLBACK_THRESH: 1.6
  CHUNK_CORRECTION          : {'size': 1500, 'overlap': 200}
  CHUNK_TEXTBOOK            : {'size': 3000, 'overlap': 250}


---
## 1. Initialize RAG Engine

Loads BGE-M3 embeddings, ChromaDB collection, and Vertex AI model.  
Expect **30-90 s** on first run (model download / warm-up).

In [4]:
import config
import time
from rag_engine import TunisianMathRAG, QueryResult, RetrievedDoc

t0 = time.monotonic()
engine = TunisianMathRAG()
init_time = time.monotonic() - t0

print(f"\nEngine ready in {init_time:.1f}s")
print(f"Collection : {config.COLLECTION_NAME}")
print(f"Chunks     : {engine.chunk_count:,}")


2026-02-21 20:30:58,341 | INFO | rag_engine | Initializing TunisianMathRAG engine...
2026-02-21 20:30:58,373 | INFO | chromadb.telemetry.product.posthog | Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-02-21 20:30:58,649 | INFO | rag_engine | Loading BGE-M3 embedding model (fp16=False)...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

2026-02-21 20:31:11,458 | INFO | FlagEmbedding.finetune.embedder.encoder_only.m3.runner | loading existing colbert_linear and sparse_linear---------
2026-02-21 20:31:11,476 | INFO | rag_engine | Embedding model loaded in 12.8s
2026-02-21 20:31:11,757 | INFO | rag_engine | Collection 'bac_math_exercises' loaded: 1890 chunks
/home/jupyter/.local/lib/python3.10/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
2026-02-21 20:31:11,762 | INFO | rag_engine | Vertex AI model: gemini-2.0-flash-exp
2026-02-21 20:31:11,763 | INFO | rag_engine | RAG engine ready in 13.4s



Engine ready in 13.4s
Collection : bac_math_exercises
Chunks     : 1,890


In [9]:
import os

DB_PATH = "./tunisian_math_db"
print("CWD:", os.getcwd())
print("DB exists:", os.path.isdir(DB_PATH))

# show what's inside (no changes)
for root, dirs, files in os.walk(DB_PATH):
    print(root)
    if dirs:  print("  dirs:", dirs[:10])
    if files: print("  files:", files[:10])
    break  # only top-level


CWD: /home/jupyter/tunisian-bac-math-rag
DB exists: True
./tunisian_math_db
  dirs: ['ab88ffbd-d994-4bb7-a530-2ee6d41f0ce5']
  files: ['chroma.sqlite3', 'index_manifest.json']


In [5]:
import chromadb

DB_PATH = "./tunisian_math_db"
client = chromadb.PersistentClient(path=DB_PATH)

cols = client.list_collections()
print("Collections found:", [c.name for c in cols])

for c in cols:
    col = client.get_collection(c.name)
    print(f"- {c.name}: count={col.count()}")


Collections found: ['bac_math_exercises']
- bac_math_exercises: count=1890


---
## 2. Database Metadata Profile 

Sample the collection to understand what is indexed:  
distribution by `type`, `chapter`, `is_solution`.

In [6]:
from collections import Counter

# ChromaDB .get() with limit; grab a meaningful sample
SAMPLE_SIZE = min(2000, engine.chunk_count)
sample = engine.collection.get(limit=SAMPLE_SIZE, include=["metadatas"])
metas = sample["metadatas"] or []

print(f"Sampled {len(metas):,} chunks out of {engine.chunk_count:,} total.\n")

type_dist    = Counter(m.get("type", "??") for m in metas)
sol_dist     = Counter(m.get("is_solution", "??") for m in metas)
chapter_dist = Counter(m.get("chapter", "??") for m in metas)
year_dist    = Counter(m.get("year", "") for m in metas)

print("=== Document type distribution ===")
for t, c in type_dist.most_common():
    pct = 100 * c / len(metas)
    bar = '#' * int(pct / 2)
    print(f"  {t:18s} {c:5d}  ({pct:5.1f}%)  {bar}")

print("\n=== is_solution distribution ===")
for s, c in sol_dist.most_common():
    pct = 100 * c / len(metas)
    print(f"  {s:18s} {c:5d}  ({pct:5.1f}%)")

print("\n=== Chapter distribution (top 15) ===")
for ch, c in chapter_dist.most_common(15):
    pct = 100 * c / len(metas)
    print(f"  {ch:35s} {c:5d}  ({pct:5.1f}%)")

# Years with content
years_with_data = sorted([y for y in year_dist if y], key=lambda y: int(y) if y.isdigit() else 0)
print(f"\n=== Years with data ({len(years_with_data)}) ===")
if years_with_data:
    print(f"  Range: {years_with_data[0]} — {years_with_data[-1]}")
    print(f"  All  : {', '.join(years_with_data)}")

Sampled 1,890 chunks out of 1,890 total.

=== Document type distribution ===
  cours               1245  ( 65.9%)  ################################
  bac_officiel         574  ( 30.4%)  ###############
  serie                 36  (  1.9%)  
  exercice              35  (  1.9%)  

=== is_solution distribution ===
  false               1280  ( 67.7%)
  true                 610  ( 32.3%)

=== Chapter distribution (top 15) ===
  Textbook Data                        1222  ( 64.7%)
  Bac avec corrections Images           574  ( 30.4%)
  Nombres complexes                      59  (  3.1%)
  Venv                                   35  (  1.9%)

=== Years with data (8) ===
  Range: 2010 — 2017
  All  : 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017


In [7]:
from rag_engine import TunisianMathRAG

engine = TunisianMathRAG()

# Test the exact filter that was crashing
selected, fp, sp, case = engine._two_stage_retrieve(
    "Déterminer le module et un argument de z = 1 + i"
)

print(f"First pass:  {len(fp)} docs")    # Should be > 0 (not 0)
print(f"Second pass: {len(sp)} docs")    # > 0 if case B, [] if case A
print(f"Selected:    {len(selected)} docs")
print(f"Case:        {case}")

# Verify: NO "Error finding id" in output
# Verify: first pass count reflects actual corrections in your DB
# Verify: if your DB is mostly "cours", first pass may be 0 and
#          second pass picks them up — that's correct behavior

for d in selected[:3]:
    m = d.metadata
    print(f"  dist={d.distance:.4f} type={m.get('type')} "
          f"ch={m.get('chapter')} sol={m.get('is_solution')}")


2026-02-21 20:31:12,662 | INFO | rag_engine | Initializing TunisianMathRAG engine...
2026-02-21 20:31:12,669 | INFO | rag_engine | Collection 'bac_math_exercises' loaded: 1890 chunks
2026-02-21 20:31:12,670 | INFO | rag_engine | Vertex AI model: gemini-2.0-flash-exp
2026-02-21 20:31:12,671 | INFO | rag_engine | RAG engine ready in 0.0s
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
2026-02-21 20:31:13,336 | INFO | rag_engine | Post-filter: 50 raw → 10 matched (requested 10), best_dist=0.7596
2026-02-21 20:31:13,337 | INFO | rag_engine | First pass (corrections): 10 docs, best_dist=0.760


First pass:  10 docs
Second pass: 0 docs
Selected:    6 docs
Case:        A
  dist=0.7596 type=serie ch=Nombres complexes sol=true
  dist=0.8243 type=bac_officiel ch=Bac avec corrections Images sol=true
  dist=0.8294 type=bac_officiel ch=Bac avec corrections Images sol=true


---
## 3. Retrieval-Only Tests (FREE — no Gemini API cost)

These call `engine._two_stage_retrieve()` which runs:
1. **First pass**: corrections / bac / series with `is_solution=true`
2. **Second pass** (if needed): textbook / cours
3. **Unfiltered fallback** (if both empty)

This uses only local embeddings + ChromaDB. **Zero API cost.**  
Use this to validate retrieval quality and tune thresholds.

In [8]:
# ── Test query bank ──────────────────────────────────────────
# Covers the main chapters of the Tunisian Bac Math curriculum.
# Each tuple: (query, expected_chapter_hint, description)

RETRIEVAL_QUERIES = [
    # Nombres complexes
    ("Déterminer le module et un argument de z = 1 + i\u221A3",
     "Nombres complexes", "Module & argument — core complex numbers"),

    # Suites numériques
    ("Montrer que la suite (Un) définie par Un+1 = f(Un) est convergente",
     "Suites", "Convergence of recursive sequence"),

    # Limites et continuité
    ("Calculer la limite de (sin x)/x quand x tend vers 0",
     "Limites", "Classic limit — sinx/x"),

    # Dérivabilité
    ("Étudier la dérivabilité de f en 0 et interpréter graphiquement",
     "Dérivabilité", "Differentiability study at a point"),

    # Intégration
    ("Calculer l'intégrale de 0 à pi/2 de x sin(x) dx par intégration par parties",
     "Intégration", "Integration by parts"),

    # Probabilités
    ("On lance un dé équilibré 3 fois. Calculer la probabilité d'obtenir exactement 2 six.",
     "Probabilités", "Binomial probability"),

    # Arithmétique
    ("Montrer que pour tout n entier naturel, n(n+1)(n+2) est divisible par 6",
     "Arithmétique", "Divisibility proof"),

    # Équations différentielles
    ("Résoudre l'équation différentielle y' + 2y = e^(-x)",
     "Équations différentielles", "First-order linear ODE"),

    # Géométrie dans l'espace
    ("Déterminer une équation cartésienne du plan passant par A, B et C",
     "Géométrie", "Cartesian equation of a plane"),

    # Logarithme / Exponentielle
    ("Résoudre dans R l'équation ln(x+1) + ln(x-1) = ln(3)",
     "Logarithme", "Logarithmic equation"),
]

In [9]:
# ── Run retrieval-only on all queries ────────────────────────
import time

retrieval_results = []

for i, (query, expected_ch, desc) in enumerate(RETRIEVAL_QUERIES, 1):
    print(f"\n{'='*75}")
    print(f"  QUERY {i}/{len(RETRIEVAL_QUERIES)}: {desc}")
    print(f"  Q: {query}")
    print(f"  Expected chapter hint: {expected_ch}")
    print(f"{'='*75}")

    t0 = time.monotonic()
    selected, first_pass, second_pass, case = engine._two_stage_retrieve(query)
    elapsed = time.monotonic() - t0

    # Build context to compute confidence
    ctx = engine._build_context(selected)
    confidence = engine._confidence_level(selected, ctx)

    best_dist = selected[0].distance if selected else None
    top_chapter = selected[0].metadata.get("chapter", "") if selected else ""
    top_type = selected[0].metadata.get("type", "") if selected else ""

    print(f"\n  Case         : {case}")
    print(f"  Confidence   : {confidence}")
    print(f"  First pass   : {len(first_pass)} docs")
    print(f"  Second pass  : {len(second_pass)} docs")
    print(f"  Selected     : {len(selected)} docs")
    print(f"  Best distance: {best_dist}")
    print(f"  Top chapter  : {top_chapter}")
    print(f"  Time         : {elapsed:.3f}s")

    print(f"\n  Selected documents:")
    for d in selected:
        m = d.metadata
        print(f"    Rank {d.rank:2d} | dist={d.distance:.4f} | "
              f"type={m.get('type',''):15s} | "
              f"ch={m.get('chapter',''):25s} | "
              f"yr={m.get('year',''):4s} | "
              f"sol={m.get('is_solution',''):5s} | "
              f"file={m.get('filename','')}")
        # Show first 120 chars of content
        snippet = d.content[:120].replace('\n', ' ')
        print(f"           Snippet: {snippet}...")

    # Accumulate for summary table
    retrieval_results.append({
        "idx": i,
        "description": desc,
        "query": query,
        "expected_chapter": expected_ch,
        "case": case,
        "confidence": confidence,
        "n_selected": len(selected),
        "n_first_pass": len(first_pass),
        "n_second_pass": len(second_pass),
        "best_distance": round(best_dist, 4) if best_dist is not None else None,
        "top_chapter": top_chapter,
        "top_type": top_type,
        "retrieval_time_s": round(elapsed, 3),
    })

print(f"\n\nRetrieval-only tests complete: {len(retrieval_results)} queries processed.")


  QUERY 1/10: Module & argument — core complex numbers
  Q: Déterminer le module et un argument de z = 1 + i√3
  Expected chapter hint: Nombres complexes


2026-02-21 20:31:13,873 | INFO | rag_engine | Post-filter: 50 raw → 10 matched (requested 10), best_dist=0.7341
2026-02-21 20:31:13,874 | INFO | rag_engine | First pass (corrections): 10 docs, best_dist=0.734



  Case         : A
  Confidence   : moyen
  First pass   : 10 docs
  Second pass  : 0 docs
  Selected     : 6 docs
  Best distance: 0.7341470122337341
  Top chapter  : Bac avec corrections Images
  Time         : 0.508s

  Selected documents:
    Rank  1 | dist=0.7341 | type=bac_officiel    | ch=Bac avec corrections Images | yr=2012 | sol=true  | file=Page_1.tex
           Snippet: .  On en déduit que $z_Q = \left(\frac{1}{2} - i\frac{\sqrt{3}}{2}\right) (ia + a - a) + a = a\left(1 + \frac{\sqrt{3}}{...
    Rank  2 | dist=0.7812 | type=bac_officiel    | ch=Bac avec corrections Images | yr=2012 | sol=true  | file=Page_1.tex
           Snippet: que :  \[ z' = -\frac{3}{2}iz + \frac{13}{2}i \]  \end{enumerate}   \item  \begin{enumerate}  \item Notons \(x\) l'absci...
    Rank  3 | dist=0.8495 | type=bac_officiel    | ch=Bac avec corrections Images | yr=2010 | sol=true  | file=Page_1.tex
           Snippet: tenu du fait que $\alpha \notin \mathbb{R}$, les affixes des points $M$ recherchés

2026-02-21 20:31:14,409 | INFO | rag_engine | Post-filter: 50 raw → 7 matched (requested 10), best_dist=0.7250
2026-02-21 20:31:14,410 | INFO | rag_engine | First pass (corrections): 7 docs, best_dist=0.725



  Case         : A
  Confidence   : fort
  First pass   : 7 docs
  Second pass  : 0 docs
  Selected     : 6 docs
  Best distance: 0.7249764800071716
  Top chapter  : Bac avec corrections Images
  Time         : 0.536s

  Selected documents:
    Rank  1 | dist=0.7250 | type=bac_officiel    | ch=Bac avec corrections Images | yr=2011 | sol=true  | file=Page_2.tex
           Snippet: _p$.  Par suite, la suite $(u_p)$ est décroissante. De plus, elle est minorée par $1$ (car $u_p \in ]1,p[$).  Toute suit...
    Rank  2 | dist=0.7680 | type=bac_officiel    | ch=Bac avec corrections Images | yr=2011 | sol=true  | file=Page_2.tex
           Snippet: )$ est décroissante et qu'elle est convergente.  \item Montrer que $\dfrac{\ln u_p}{u_p} = \dfrac{1}{p}$. En déduire la ...
    Rank  3 | dist=0.8453 | type=bac_officiel    | ch=Bac avec corrections Images | yr=2011 | sol=true  | file=Page_2.tex
           Snippet: sur l'axe des abscisses les termes $u_3$, $u_4$, $u_5$ et $u_6$ de la suite $(u_p)$.

...


  Case         : A
  Confidence   : fort
  First pass   : 10 docs
  Second pass  : 0 docs
  Selected     : 6 docs
  Best distance: 0.7881342172622681
  Top chapter  : Bac avec corrections Images
  Time         : 0.539s

  Selected documents:
    Rank  1 | dist=0.7881 | type=bac_officiel    | ch=Bac avec corrections Images | yr=2017 | sol=true  | file=Page_2.tex
           Snippet: {1}{1+(f(x))^2}\right) = 2f'(x)\left(1 - \frac{1}{1+e^x - 1}\right) \\ &= 2 \cdot \frac{e^x}{2\sqrt{e^x - 1}} \left(\fra...
    Rank  2 | dist=0.7968 | type=bac_officiel    | ch=Bac avec corrections Images | yr=2012 | sol=true  | file=Page_1.tex
           Snippet: n divisant par $i$ (puisque $i \neq 0$), nous obtenons :  $y = -\frac{3}{2}x + \frac{13}{2}$.  Multipliant par 2, il en ...
    Rank  3 | dist=0.7994 | type=bac_officiel    | ch=Bac avec corrections Images | yr=2014 | sol=true  | file=Page_3.tex
           Snippet: } y}$ avec  $f^{-1}(x)=y, y \in \left]-\frac{\pi}{4}, \frac{\pi}{2}\right[$ et $x \

2026-02-21 20:31:18,061 | INFO | rag_engine | Post-filter: 50 raw → 10 matched (requested 10), best_dist=0.8085
2026-02-21 20:31:18,062 | INFO | rag_engine | First pass (corrections): 10 docs, best_dist=0.808



  Case         : A
  Confidence   : fort
  First pass   : 10 docs
  Second pass  : 0 docs
  Selected     : 6 docs
  Best distance: 0.8084768652915955
  Top chapter  : Bac avec corrections Images
  Time         : 0.524s

  Selected documents:
    Rank  1 | dist=0.8085 | type=bac_officiel    | ch=Bac avec corrections Images | yr=2014 | sol=true  | file=Page_1.tex
           Snippet: hère $S$ est tangente à chacun des deux plans $P$ et $Q$ respectivement en $A$ et $J$.  \noindent 3) a) $A' = t(A) \Left...
    Rank  2 | dist=0.8271 | type=bac_officiel    | ch=Bac avec corrections Images | yr=2015 | sol=true  | file=Page_1.tex
           Snippet: u plan limitée par les deux courbes $(C_f)$ et $(C_g)$ et les droites d'équations $x=0$ et $x=1$.  \end{enumerate}  \end...
    Rank  3 | dist=0.8665 | type=bac_officiel    | ch=Bac avec corrections Images | yr=2011 | sol=true  | file=Page_1.tex
           Snippet: Voici la transcription LaTeX de l'énoncé de l'exercice :  ```latex \documentclass{a

2026-02-21 20:31:18,614 | INFO | rag_engine | Post-filter: 50 raw → 10 matched (requested 10), best_dist=0.7973
2026-02-21 20:31:18,615 | INFO | rag_engine | First pass (corrections): 10 docs, best_dist=0.797



  Case         : A
  Confidence   : fort
  First pass   : 10 docs
  Second pass  : 0 docs
  Selected     : 6 docs
  Best distance: 0.7973203659057617
  Top chapter  : Bac avec corrections Images
  Time         : 0.552s

  Selected documents:
    Rank  1 | dist=0.7973 | type=bac_officiel    | ch=Bac avec corrections Images | yr=2017 | sol=true  | file=Page_2.tex
           Snippet: n)$ et $x = \ln(n+1)$. Montrer que $A_n = 2 \sqrt{n} \, g^{-1} \left( \frac{1}{\sqrt{n}} \right) + \ln \left( \frac{n}{n...
    Rank  2 | dist=0.8731 | type=bac_officiel    | ch=Bac avec corrections Images | yr=2017 | sol=true  | file=Page_2.tex
           Snippet: Here is the transcription of the mathematical text and formulas from the image into LaTeX format.  **d)** $T : y = f'(\a...
    Rank  3 | dist=0.8741 | type=bac_officiel    | ch=Bac avec corrections Images | yr=2014 | sol=true  | file=Page_2.tex
           Snippet: Based on the image provided, here is the transcription of the mathematical text and

### 3.1 Retrieval Summary Table (thesis-ready)

Compact view of all retrieval results.  
Screenshot this for your thesis report.

In [6]:
# ── Summary table ────────────────────────────────────────────

hdr = (f"{'#':>2s}  {'Description':<30s}  {'Case':>4s}  {'Conf':>6s}  "
       f"{'Docs':>4s}  {'BestD':>7s}  {'TopChapter':<25s}  {'TopType':<15s}  {'Time':>6s}")
sep = '-' * len(hdr)

print(sep)
print(hdr)
print(sep)

for r in retrieval_results:
    desc_short = r['description'][:28] + '..' if len(r['description']) > 30 else r['description']
    bd = f"{r['best_distance']:.4f}" if r['best_distance'] is not None else '  N/A '
    print(f"{r['idx']:2d}  {desc_short:<30s}  {r['case']:>4s}  {r['confidence']:>6s}  "
          f"{r['n_selected']:4d}  {bd:>7s}  {r['top_chapter']:<25s}  {r['top_type']:<15s}  "
          f"{r['retrieval_time_s']:6.3f}")

print(sep)

# Aggregate stats
avg_time = sum(r['retrieval_time_s'] for r in retrieval_results) / len(retrieval_results)
case_a = sum(1 for r in retrieval_results if r['case'] == 'A')
case_b = sum(1 for r in retrieval_results if r['case'] == 'B')
dists = [r['best_distance'] for r in retrieval_results if r['best_distance'] is not None]
avg_dist = sum(dists) / len(dists) if dists else 0

print(f"\nAggregate:")
print(f"  Case A (correction match) : {case_a}/{len(retrieval_results)}")
print(f"  Case B (textbook fallback): {case_b}/{len(retrieval_results)}")
print(f"  Avg best distance         : {avg_dist:.4f}")
print(f"  Avg retrieval time        : {avg_time:.3f}s")

-------------------------------------------------------------------------------------------------------------------
 #  Description                     Case    Conf  Docs    BestD  TopChapter                 TopType            Time
-------------------------------------------------------------------------------------------------------------------
 1  Module & argument — core com..     A   moyen     6   0.7341  Bac avec corrections Images  bac_officiel      0.580
 2  Convergence of recursive seq..     A    fort     6   0.7250  Bac avec corrections Images  bac_officiel      0.637
 3  Classic limit — sinx/x             A    fort     6   0.8174  Bac avec corrections Images  bac_officiel      0.590
 4  Differentiability study at a..     A    fort     6   0.7231  Bac avec corrections Images  bac_officiel      0.583
 5  Integration by parts               A   moyen     6   0.8335  Bac avec corrections Images  bac_officiel      0.637
 6  Binomial probability               A    fort     6   0.958

---
## 4. Full RAG Queries (Gemini API cost — targeted subset)

Runs `engine.query()` which calls Vertex AI Gemini.  
We select **4 representative queries** to validate the full pipeline  
(retrieval + prompt compilation + generation) while keeping API cost low.

Each result captures: answer text, retrieval case, confidence, timings, sources.

In [10]:
# ── Full RAG test queries ────────────────────────────────────
# Deliberately small set to control API cost.
# Mix of modes: correction (dry, exam-style) and coaching (pedagogical).

RAG_QUERIES = [
    {
        "question": "Résoudre dans C l'équation z^2 + 2z + 5 = 0",
        "mode": "correction",
        "desc": "Complex equation — correction mode",
    },
    {
        "question": "Comment montrer qu'une suite définie par récurrence est convergente ?",
        "mode": "coaching",
        "desc": "Recursive sequence convergence — coaching mode",
    },
    {
        "question": "Calculer l'intégrale de 0 à 1 de x*e^x dx",
        "mode": "correction",
        "desc": "Integration by parts — correction mode",
    },
    {
        "question": "On considère la variable aléatoire X suivant la loi binomiale B(10, 0.3). Calculer P(X=3) et E(X).",
        "mode": "coaching",
        "desc": "Binomial distribution — coaching mode",
    },
]

In [11]:
# ── Execute full RAG pipeline ────────────────────────────────
import time
from rag_engine import TunisianMathRAG
engine = TunisianMathRAG()
print(f"Chunks: {engine.chunk_count}")

full_results = []

for i, q in enumerate(RAG_QUERIES, 1):
    print(f"\n{'='*75}")
    print(f"  FULL RAG QUERY {i}/{len(RAG_QUERIES)}: {q['desc']}")
    print(f"  Mode: {q['mode']}")
    print(f"  Q: {q['question']}")
    print(f"{'='*75}\n")

    result = engine.query(q["question"], mode=q["mode"])

    # ── Retrieval info ──
    print(f"  Retrieval case  : {result.retrieval_case}")
    print(f"  Confidence      : {result.confidence}")
    print(f"  Selected docs   : {len(result.selected_docs)}")
    print(f"  First pass docs : {len(result.first_pass_docs)}")
    print(f"  Second pass docs: {len(result.second_pass_docs)}")
    print(f"  Retrieval time  : {result.retrieval_time:.3f}s")
    print(f"  Generation time : {result.generation_time:.3f}s")
    print(f"  Total time      : {result.total_time:.3f}s")

    if result.error:
        print(f"  ERROR: {result.error}")

    # ── Retrieved sources ──
    print(f"\n  Sources used:")
    for d in result.selected_docs:
        m = d.metadata
        print(f"    Rank {d.rank} | dist={d.distance:.4f} | "
              f"type={m.get('type','')} | ch={m.get('chapter','')} | "
              f"yr={m.get('year','')} | sol={m.get('is_solution','')}")

    # ── Answer ──
    print(f"\n  {'─'*60}")
    print(f"  ANSWER:")
    print(f"  {'─'*60}")
    if result.answer:
        # Print full answer (may be long)
        for line in result.answer.split('\n'):
            print(f"  {line}")
    else:
        print(f"  (no answer — error: {result.error})")

    full_results.append({
        "idx": i,
        "desc": q["desc"],
        "mode": q["mode"],
        "case": result.retrieval_case,
        "confidence": result.confidence,
        "n_selected": len(result.selected_docs),
        "best_distance": round(result.selected_docs[0].distance, 4) if result.selected_docs else None,
        "retrieval_time": round(result.retrieval_time, 3),
        "generation_time": round(result.generation_time, 3),
        "total_time": round(result.total_time, 3),
        "has_answer": bool(result.answer),
        "error": result.error,
        "answer_length": len(result.answer) if result.answer else 0,
        "answer_preview": (result.answer or "")[:150].replace('\n', ' '),
    })

print(f"\n\nFull RAG tests complete: {len(full_results)} queries.")

2026-02-21 20:31:18,640 | INFO | rag_engine | Initializing TunisianMathRAG engine...
2026-02-21 20:31:18,648 | INFO | rag_engine | Collection 'bac_math_exercises' loaded: 1890 chunks
2026-02-21 20:31:18,650 | INFO | rag_engine | Vertex AI model: gemini-2.0-flash-exp
2026-02-21 20:31:18,651 | INFO | rag_engine | RAG engine ready in 0.0s


Chunks: 1890

  FULL RAG QUERY 1/4: Complex equation — correction mode
  Mode: correction
  Q: Résoudre dans C l'équation z^2 + 2z + 5 = 0



2026-02-21 20:31:19,214 | INFO | rag_engine | Post-filter: 50 raw → 10 matched (requested 10), best_dist=0.8339
2026-02-21 20:31:19,215 | INFO | rag_engine | First pass (corrections): 10 docs, best_dist=0.834
2026-02-21 20:31:19,216 | INFO | rag_engine | Retrieval: case=A confidence=fort selected=6 in 0.56s
2026-02-21 20:31:25,124 | INFO | rag_engine | Query done: retrieval=0.56s generation=5.91s total=6.47s


  Retrieval case  : A
  Confidence      : fort
  Selected docs   : 6
  First pass docs : 10
  Second pass docs: 0
  Retrieval time  : 0.561s
  Generation time : 5.907s
  Total time      : 6.469s

  Sources used:
    Rank 1 | dist=0.8339 | type=bac_officiel | ch=Bac avec corrections Images | yr=2016 | sol=true
    Rank 2 | dist=0.8789 | type=bac_officiel | ch=Bac avec corrections Images | yr=2015 | sol=true
    Rank 3 | dist=0.9065 | type=bac_officiel | ch=Bac avec corrections Images | yr=2017 | sol=true
    Rank 4 | dist=0.9071 | type=bac_officiel | ch=Bac avec corrections Images | yr=2013 | sol=true
    Rank 5 | dist=0.9114 | type=bac_officiel | ch=Bac avec corrections Images | yr=2013 | sol=true
    Rank 6 | dist=0.9142 | type=bac_officiel | ch=Bac avec corrections Images | yr=2016 | sol=true

  ────────────────────────────────────────────────────────────
  ANSWER:
  ────────────────────────────────────────────────────────────
  Ahla w sahla ! Equation fel C, c'est du pain quotidien 

2026-02-21 20:31:26,064 | INFO | rag_engine | Post-filter: 50 raw → 3 matched (requested 10), best_dist=0.8148
2026-02-21 20:31:26,065 | INFO | rag_engine | First pass (corrections): 3 docs, best_dist=0.815
2026-02-21 20:31:26,066 | INFO | rag_engine | Retrieval: case=A confidence=moyen selected=3 in 0.94s
2026-02-21 20:31:34,455 | INFO | rag_engine | Query done: retrieval=0.94s generation=8.39s total=9.33s


  Retrieval case  : A
  Confidence      : moyen
  Selected docs   : 3
  First pass docs : 3
  Second pass docs: 0
  Retrieval time  : 0.940s
  Generation time : 8.389s
  Total time      : 9.330s

  Sources used:
    Rank 1 | dist=0.8148 | type=bac_officiel | ch=Bac avec corrections Images | yr=2011 | sol=true
    Rank 2 | dist=0.8452 | type=bac_officiel | ch=Bac avec corrections Images | yr=2011 | sol=true
    Rank 3 | dist=0.9193 | type=bac_officiel | ch=Bac avec corrections Images | yr=2011 | sol=true

  ────────────────────────────────────────────────────────────
  ANSWER:
  ────────────────────────────────────────────────────────────
  Ahla s7aybi, convergence suite, 9allek 5ayef! Tawa nwarrik kifach.
  
  1) **Identification** : Convergence d'une suite définie par récurrence.
  
  2) **Rappel du cours** :
  Pour montrer qu'une suite $(u_n)$ est convergente, on peut utiliser les théorèmes suivants (qui ne sont pas explicitement dans les sources, mais font partie du cours classique 

2026-02-21 20:31:34,947 | INFO | rag_engine | Post-filter: 50 raw → 10 matched (requested 10), best_dist=0.8522
2026-02-21 20:31:34,948 | INFO | rag_engine | First pass (corrections): 10 docs, best_dist=0.852
2026-02-21 20:31:34,949 | INFO | rag_engine | Retrieval: case=A confidence=moyen selected=6 in 0.49s
2026-02-21 20:31:39,931 | INFO | rag_engine | Query done: retrieval=0.49s generation=4.98s total=5.47s


  Retrieval case  : A
  Confidence      : moyen
  Selected docs   : 6
  First pass docs : 10
  Second pass docs: 0
  Retrieval time  : 0.492s
  Generation time : 4.982s
  Total time      : 5.474s

  Sources used:
    Rank 1 | dist=0.8522 | type=bac_officiel | ch=Bac avec corrections Images | yr=2011 | sol=true
    Rank 2 | dist=0.8652 | type=bac_officiel | ch=Bac avec corrections Images | yr=2011 | sol=true
    Rank 3 | dist=0.8855 | type=bac_officiel | ch=Bac avec corrections Images | yr=2014 | sol=true
    Rank 4 | dist=0.8857 | type=bac_officiel | ch=Bac avec corrections Images | yr=2015 | sol=true
    Rank 5 | dist=0.8972 | type=bac_officiel | ch=Bac avec corrections Images | yr=2010 | sol=true
    Rank 6 | dist=0.9002 | type=bac_officiel | ch=Bac avec corrections Images | yr=2013 | sol=true

  ────────────────────────────────────────────────────────────
  ANSWER:
  ────────────────────────────────────────────────────────────
  Ahla! Pas de soucis, 7aja sahla barcha. Nfehmouk comme

2026-02-21 20:31:40,539 | INFO | rag_engine | Post-filter: 50 raw → 10 matched (requested 10), best_dist=0.7453
2026-02-21 20:31:40,540 | INFO | rag_engine | First pass (corrections): 10 docs, best_dist=0.745
2026-02-21 20:31:40,541 | INFO | rag_engine | Retrieval: case=A confidence=fort selected=6 in 0.61s
2026-02-21 20:31:41,058 | WARNING | rag_engine | Generation attempt 1/3 failed: 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.. Sleeping 2s
2026-02-21 20:31:43,914 | WARNING | rag_engine | Generation attempt 2/3 failed: 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.. Sleeping 4s
2026-02-21 20:31:48,208 | WARNING | rag_engine | Generation attempt 3/3 failed: 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error

  Retrieval case  : A
  Confidence      : fort
  Selected docs   : 6
  First pass docs : 10
  Second pass docs: 0
  Retrieval time  : 0.608s
  Generation time : 15.676s
  Total time      : 16.285s
  ERROR: Generation failed after 3 retries: 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.

  Sources used:
    Rank 1 | dist=0.7453 | type=bac_officiel | ch=Bac avec corrections Images | yr=2012 | sol=true
    Rank 2 | dist=0.8893 | type=bac_officiel | ch=Bac avec corrections Images | yr=2014 | sol=true
    Rank 3 | dist=0.8895 | type=bac_officiel | ch=Bac avec corrections Images | yr=2012 | sol=true
    Rank 4 | dist=0.8918 | type=bac_officiel | ch=Bac avec corrections Images | yr=2011 | sol=true
    Rank 5 | dist=0.8991 | type=bac_officiel | ch=Bac avec corrections Images | yr=2014 | sol=true
    Rank 6 | dist=0.9011 | type=bac_officiel | ch=Bac avec corrections Images | yr=2011 | sol=tr

### 4.1 Full RAG Summary Table

In [16]:
hdr2 = (f"{'#':>2s}  {'Description':<35s}  {'Mode':<11s}  {'Case':>4s}  {'Conf':>6s}  "
        f"{'Docs':>4s}  {'BestD':>7s}  {'RetT':>5s}  {'GenT':>5s}  {'TotT':>5s}  "
        f"{'Ans':>5s}  {'Preview':<40s}")
sep2 = '-' * len(hdr2)

print(sep2)
print(hdr2)
print(sep2)

for r in full_results:
    desc_s = r['desc'][:33] + '..' if len(r['desc']) > 35 else r['desc']
    bd = f"{r['best_distance']:.4f}" if r['best_distance'] is not None else '  N/A '
    ok = 'OK' if r['has_answer'] else 'FAIL'
    preview = r['answer_preview'][:38] + '..' if len(r['answer_preview']) > 40 else r['answer_preview']
    print(f"{r['idx']:2d}  {desc_s:<35s}  {r['mode']:<11s}  {r['case']:>4s}  {r['confidence']:>6s}  "
          f"{r['n_selected']:4d}  {bd:>7s}  {r['retrieval_time']:5.2f}  {r['generation_time']:5.2f}  "
          f"{r['total_time']:5.2f}  {ok:>5s}  {preview:<40s}")

print(sep2)

successes = sum(1 for r in full_results if r['has_answer'])
print(f"\n  Success rate: {successes}/{len(full_results)}")
if full_results:
    avg_gen = sum(r['generation_time'] for r in full_results) / len(full_results)
    avg_tot = sum(r['total_time'] for r in full_results) / len(full_results)
    print(f"  Avg generation time: {avg_gen:.2f}s")
    print(f"  Avg total time     : {avg_tot:.2f}s")

-------------------------------------------------------------------------------------------------------------------------------------------------------
 #  Description                          Mode         Case    Conf  Docs    BestD   RetT   GenT   TotT    Ans  Preview                                 
-------------------------------------------------------------------------------------------------------------------------------------------------------
 1  Complex equation — correction mode   correction      B    fort     6   0.8974   2.37   8.51  10.87     OK  Ahla s7abi, équation simple, matkhafou..
 2  Recursive sequence convergence — ..  coaching        B    fort     6   1.0681   3.28   5.26   8.54     OK  Ahla sidi, c'est une question classiqu..
 3  Integration by parts — correction..  correction      B    fort     6   0.9862   2.21  19.78  21.99     OK  Ahla s7aybi, intégration par parties c..
 4  Binomial distribution — coaching ..  coaching        B   moyen     6   0.9337   3.90

---
## 5. Syllabus Guard Test

The system prompt forbids out-of-program methods (L'Hopital, Taylor series, etc.).  
Verify that the engine **refuses** and proposes a bac-compatible alternative.

In [17]:
GUARD_QUERIES = [
    "Utilise la règle de L'Hôpital pour calculer lim(x->0) sin(x)/x",
    "Diagonalise la matrice A pour calculer A^n",
]

for gq in GUARD_QUERIES:
    print(f"\n{'='*75}")
    print(f"  SYLLABUS GUARD TEST")
    print(f"  Q: {gq}")
    print(f"{'='*75}\n")

    result = engine.query(gq, mode="correction")

    print(f"  Case: {result.retrieval_case} | Confidence: {result.confidence}")
    print(f"  Time: retrieval={result.retrieval_time:.2f}s generation={result.generation_time:.2f}s")
    print(f"\n  ANSWER (first 800 chars):")
    print(f"  {'─'*60}")
    answer_text = result.answer if result.answer else f"ERROR: {result.error}"
    for line in answer_text[:800].split('\n'):
        print(f"  {line}")
    if len(answer_text) > 800:
        print(f"  [...truncated, total {len(answer_text)} chars]")
    print()


  SYLLABUS GUARD TEST
  Q: Utilise la règle de L'Hôpital pour calculer lim(x->0) sin(x)/x



2026-02-13 09:33:06,114 | WARNING | rag_engine | Retrieval failed (filter={'$and': [{'type': {'$in': ['bac_officiel', 'serie', 'devoir']}}, {'is_solution': 'true'}]}): Error executing plan: Internal error: Error finding id
2026-02-13 09:33:06,116 | INFO | rag_engine | First pass (corrections): 0 docs, best_dist=inf
2026-02-13 09:33:07,111 | WARNING | rag_engine | Retrieval failed (filter={'type': {'$in': ['cours', 'textbook']}}): Error executing plan: Internal error: Error finding id
2026-02-13 09:33:07,112 | INFO | rag_engine | Second pass (textbook): 0 docs, best_dist=inf
2026-02-13 09:33:07,114 | INFO | rag_engine | Both passes empty — unfiltered fallback
2026-02-13 09:33:08,025 | INFO | rag_engine | Retrieval: case=B confidence=fort selected=6 in 3.29s
2026-02-13 09:33:09,310 | WARNING | rag_engine | Generation attempt 1/3 failed: 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.. S

  Case: B | Confidence: fort
  Time: retrieval=3.29s generation=16.16s

  ANSWER (first 800 chars):
  ────────────────────────────────────────────────────────────
  ERROR: Generation failed after 3 retries: 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.


  SYLLABUS GUARD TEST
  Q: Diagonalise la matrice A pour calculer A^n



2026-02-13 09:33:25,205 | WARNING | rag_engine | Retrieval failed (filter={'$and': [{'type': {'$in': ['bac_officiel', 'serie', 'devoir']}}, {'is_solution': 'true'}]}): Error executing plan: Internal error: Error finding id
2026-02-13 09:33:25,206 | INFO | rag_engine | First pass (corrections): 0 docs, best_dist=inf
2026-02-13 09:33:26,232 | WARNING | rag_engine | Retrieval failed (filter={'type': {'$in': ['cours', 'textbook']}}): Error executing plan: Internal error: Error finding id
2026-02-13 09:33:26,233 | INFO | rag_engine | Second pass (textbook): 0 docs, best_dist=inf
2026-02-13 09:33:26,234 | INFO | rag_engine | Both passes empty — unfiltered fallback
2026-02-13 09:33:27,242 | INFO | rag_engine | Retrieval: case=B confidence=fort selected=6 in 3.06s
2026-02-13 09:33:35,246 | INFO | rag_engine | Query done: retrieval=3.06s generation=8.00s total=11.06s


  Case: B | Confidence: fort
  Time: retrieval=3.06s generation=8.00s

  ANSWER (first 800 chars):
  ────────────────────────────────────────────────────────────
  Eeeh, صاحبي, матрица و diagonalisation, c'est pas toujours évident! Nchallah rabbi m3ak. On va voir ça ensemble خطوة بخطوة.
  
  1) **Identification** : Algèbre linéaire, diagonalisation d'une matrice.
  
  2) **Rappel du cours**:
  Le contexte fourni ne contient pas de cours sur la diagonalisation des matrices. Cependant, on peut rappeler les éléments essentiels si la diagonalisation est au programme (ce qui est possible en section Maths).
  
  3) **Méthode** :
  Si la diagonalisation est au programme, voici un plan de méthode général :
     1. Calculer le polynôme caractéristique de la matrice $A$, noté $\chi_A(\lambda) = \det(A - \lambda I)$, où $I$ est la matrice identité.
     2. Trouver les racines du polynôme caractéristique. Ces racines sont les valeurs propres de $A$.
     3. Pour chaque valeur propre 
  [...truncat

---
## 6. Retrieval Edge Cases (FREE)

Test queries that are deliberately vague, off-topic, or in Derja  
to observe how the two-stage retrieval and confidence scoring behave.

In [18]:
EDGE_QUERIES = [
    ("Kifeh na3mel exercice mte3 les nombres complexes?",
     "Derja input — should still retrieve complex number content"),
    ("What is the capital of France?",
     "Off-topic — should yield low confidence / high distance"),
    ("x",
     "Minimal input — stress test"),
    ("Bac 2015 exercice 3 nombres complexes correction",
     "Specific bac reference — should find exact match if indexed"),
]

for query, desc in EDGE_QUERIES:
    selected, fp, sp, case = engine._two_stage_retrieve(query)
    ctx = engine._build_context(selected)
    conf = engine._confidence_level(selected, ctx)
    bd = selected[0].distance if selected else None
    tc = selected[0].metadata.get('chapter', '') if selected else ''

    print(f"  [{desc}]")
    print(f"    Q: {query}")
    print(f"    Case={case}  Conf={conf}  Docs={len(selected)}  "
          f"BestDist={f'{bd:.4f}' if bd else 'N/A'}  TopCh={tc}")
    if selected:
        snip = selected[0].content[:80].replace('\n', ' ')
        print(f"    Top snippet: {snip}...")
    print()

2026-02-13 09:34:26,826 | WARNING | rag_engine | Retrieval failed (filter={'$and': [{'type': {'$in': ['bac_officiel', 'serie', 'devoir']}}, {'is_solution': 'true'}]}): Error executing plan: Internal error: Error finding id
2026-02-13 09:34:26,828 | INFO | rag_engine | First pass (corrections): 0 docs, best_dist=inf
2026-02-13 09:34:27,576 | WARNING | rag_engine | Retrieval failed (filter={'type': {'$in': ['cours', 'textbook']}}): Error executing plan: Internal error: Error finding id
2026-02-13 09:34:27,577 | INFO | rag_engine | Second pass (textbook): 0 docs, best_dist=inf
2026-02-13 09:34:27,578 | INFO | rag_engine | Both passes empty — unfiltered fallback


  [Derja input — should still retrieve complex number content]
    Q: Kifeh na3mel exercice mte3 les nombres complexes?
    Case=B  Conf=fort  Docs=6  BestDist=0.8735  TopCh=Nombres complexes
    Top snippet: Marhaba ya weldi/benti! Voici la transcription de ton cours de maths en LaTeX, c...



2026-02-13 09:34:28,869 | WARNING | rag_engine | Retrieval failed (filter={'$and': [{'type': {'$in': ['bac_officiel', 'serie', 'devoir']}}, {'is_solution': 'true'}]}): Error executing plan: Internal error: Error finding id
2026-02-13 09:34:28,870 | INFO | rag_engine | First pass (corrections): 0 docs, best_dist=inf
2026-02-13 09:34:29,429 | WARNING | rag_engine | Retrieval failed (filter={'type': {'$in': ['cours', 'textbook']}}): Error executing plan: Internal error: Error finding id
2026-02-13 09:34:29,430 | INFO | rag_engine | Second pass (textbook): 0 docs, best_dist=inf
2026-02-13 09:34:29,431 | INFO | rag_engine | Both passes empty — unfiltered fallback


  [Off-topic — should yield low confidence / high distance]
    Q: What is the capital of France?
    Case=B  Conf=moyen  Docs=6  BestDist=1.2780  TopCh=Nombres complexes
    Top snippet: Voici la transcription LaTeX de la solution :  ```latex \documentclass[12pt]{art...



2026-02-13 09:34:30,778 | WARNING | rag_engine | Retrieval failed (filter={'$and': [{'type': {'$in': ['bac_officiel', 'serie', 'devoir']}}, {'is_solution': 'true'}]}): Error executing plan: Internal error: Error finding id
2026-02-13 09:34:30,779 | INFO | rag_engine | First pass (corrections): 0 docs, best_dist=inf
2026-02-13 09:34:31,584 | WARNING | rag_engine | Retrieval failed (filter={'type': {'$in': ['cours', 'textbook']}}): Error executing plan: Internal error: Error finding id
2026-02-13 09:34:31,586 | INFO | rag_engine | Second pass (textbook): 0 docs, best_dist=inf
2026-02-13 09:34:31,587 | INFO | rag_engine | Both passes empty — unfiltered fallback


  [Minimal input — stress test]
    Q: x
    Case=B  Conf=fort  Docs=6  BestDist=1.1077  TopCh=Nombres complexes
    Top snippet: t{OM}=\sqrt{a^2+b^2}$.  \item Pour tous points M et N d'affixes $z_M$ et $z_N$, ...



2026-02-13 09:34:33,344 | WARNING | rag_engine | Retrieval failed (filter={'$and': [{'type': {'$in': ['bac_officiel', 'serie', 'devoir']}}, {'is_solution': 'true'}]}): Error executing plan: Internal error: Error finding id
2026-02-13 09:34:33,346 | INFO | rag_engine | First pass (corrections): 0 docs, best_dist=inf
2026-02-13 09:34:34,320 | WARNING | rag_engine | Retrieval failed (filter={'type': {'$in': ['cours', 'textbook']}}): Error executing plan: Internal error: Error finding id
2026-02-13 09:34:34,322 | INFO | rag_engine | Second pass (textbook): 0 docs, best_dist=inf
2026-02-13 09:34:34,323 | INFO | rag_engine | Both passes empty — unfiltered fallback


  [Specific bac reference — should find exact match if indexed]
    Q: Bac 2015 exercice 3 nombres complexes correction
    Case=B  Conf=fort  Docs=6  BestDist=0.8921  TopCh=Nombres complexes
    Top snippet: Voici la transcription LaTeX de la solution :  ```latex \documentclass[12pt]{art...



---
## 7. Combined Results Export

Save all results to a JSON file for reproducibility and later analysis.

In [19]:
import json
from datetime import datetime, timezone

export = {
    "metadata": {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "python_version": sys.version,
        "chunk_count": engine.chunk_count,
        "config": {
            "CHAT_MODEL_ID": config.CHAT_MODEL_ID,
            "EMBEDDING_MODEL": config.EMBEDDING_MODEL_NAME,
            "USE_FP16": config.USE_FP16,
            "RETRIEVE_K_FIRST_PASS": config.RETRIEVE_K_FIRST_PASS,
            "RETRIEVE_K_SECOND_PASS": config.RETRIEVE_K_SECOND_PASS,
            "USE_TOP_N": config.USE_TOP_N,
            "SIMILARITY_GOOD_THRESHOLD": config.SIMILARITY_GOOD_THRESHOLD,
            "SIMILARITY_FALLBACK_THRESHOLD": config.SIMILARITY_FALLBACK_THRESHOLD,
        },
    },
    "retrieval_only_results": retrieval_results,
    "full_rag_results": full_results,
}

out_path = "test_rag_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(export, f, ensure_ascii=False, indent=2)

print(f"Results saved to: {out_path}")
print(f"File size: {os.path.getsize(out_path):,} bytes")

NameError: name 'sys' is not defined

---
## 8. Final Validation Report

One-screen summary suitable for thesis screenshot.

In [20]:
print(f"""
╔══════════════════════════════════════════════════════════════════╗
║             TUNISIAN BAC MATH RAG — VALIDATION REPORT          ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                ║
║  Environment                                                   ║
║    Python          : {sys.version.split()[0]:<42s} ║
║    Embedding model : {config.EMBEDDING_MODEL_NAME:<42s} ║
║    FP16            : {str(config.USE_FP16):<42s} ║
║    LLM             : {config.CHAT_MODEL_ID:<42s} ║
║    DB chunks       : {engine.chunk_count:<42,d} ║
║                                                                ║
║  Retrieval-only tests (FREE, no API cost)                      ║
║    Queries tested   : {len(retrieval_results):<41d} ║
║    Case A (match)   : {case_a:<41d} ║
║    Case B (fallback) : {case_b:<40d} ║
║    Avg best distance : {avg_dist:<40.4f} ║
║    Avg retrieval time: {avg_time:<40.3f} ║
║                                                                ║
║  Full RAG tests (with Gemini generation)                       ║
║    Queries tested  : {len(full_results):<41d} ║
║    Successes       : {successes:<41d} ║""")

if full_results:
    print(f"""║    Avg gen time    : {avg_gen:<41.2f} ║
║    Avg total time  : {avg_tot:<41.2f} ║""")

print(f"""║                                                                ║
║  Syllabus guard   : {len(GUARD_QUERIES)} queries tested                            ║
║  Edge cases       : {len(EDGE_QUERIES)} queries tested                            ║
║                                                                ║
║  Results exported to: test_rag_results.json                    ║
║                                                                ║
╚══════════════════════════════════════════════════════════════════╝
""")

NameError: name 'sys' is not defined